# Network lab 7: An access list that keeps guests out (Network Simulator)

Write an extended access list so guests reach the internet web server but not staff computers or the file server, apply it in the right place, and test both sides.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `netbuilder.json`, `README.txt`.

## 1. Add the servers

Two servers are waiting below the switch. Cable **files01** to **Fa0/3** (a staff port) with **Copper Straight-Through** and give it 192.168.10.10, gateway 192.168.10.1. Set the **Cable** list back to **Automatic** and cable **web-out** to the router: it takes the router's first free port, **G0/1**. Give web-out 203.0.113.10, gateway 203.0.113.1; its **Web server** is already ticked. On the router CLI:

```
enable
configure terminal
interface g0/1
 ip address 203.0.113.1 255.255.255.0
 no shutdown
end
```

From guest-1's Command Prompt, type `curl 203.0.113.10` (it fetches web-out's home page) and `ping 192.168.10.10`: both work for now.

- [ ] files01 and web-out cabled and addressed
- [ ] Router G0/1 up
- [ ] Before the ACL: guest-1 reaches both

> Hint: Test before you block, so you know any failure later is caused by your rule and not by a typo.

?> Before the ACL, can guest-1 ping files01? (yes or no) :: sha256:8a798890fe93817163b10b5f7bd2ca4d25d84c52739a645a889c173eee7d9d3d

## 2. Write the access list

On the router:

```
configure terminal
access-list 120 deny ip 192.168.20.0 0.0.0.255 192.168.10.0 0.0.0.255
access-list 120 permit tcp 192.168.20.0 0.0.0.255 any eq 80
access-list 120 permit udp 192.168.20.0 0.0.0.255 any eq 53
access-list 120 deny ip any any
interface g0/0.20
 ip access-group 120 in
end
show access-lists
```

Rules are read top to bottom and the first match wins, so the deny for the staff network comes first.

- [ ] Access list 120 written in the right order
- [ ] Applied inbound on the guest subinterface
- [ ] show access-lists checked: lines 10 to 40 in order

> Hint: A wildcard mask is the opposite of a netmask: 0.0.0.255 means "any value in the last number".

?> Which wildcard mask matches a whole /24 network? :: sha256:dc177893bc79d6b9613a86ef3ed02b2327ecad455fe79484367a30be115a34fd
?> Which interface gets the list, inbound? (as typed in the command) :: sha256:6d02f277b67f8ec30c16ab1dc020c5763b39bfd9ff0816429af51dbd6d7acfe2

## 3. Test both sides

From guest-1: `curl 203.0.113.10` (allowed) and `ping 192.168.10.10` (blocked: **Destination net unreachable** from the router). From staff-1: `ping 192.168.10.10` and `curl 203.0.113.10` (both still fine). Then try `ping 203.0.113.10` from guest-1: blocked too, because the policy allows the web, not ping.

- [ ] Guest to web works
- [ ] Guest to files01 blocked
- [ ] Staff unaffected
- [ ] Guest ping to web-out blocked, and you can say which line did it

> Hint: Every allow-list ends with an invisible "deny any". Writing it yourself makes the policy plain to whoever reads it next.

?> After the ACL, can guest-1 still open the web server? (yes or no) :: sha256:8a798890fe93817163b10b5f7bd2ca4d25d84c52739a645a889c173eee7d9d3d
?> After the ACL, can guest-1 ping files01? (yes or no) :: sha256:9390298f3fb0c5b160498935d79cb139aef28e1c47358b4bbba61862b9c26e59

## 4. Where next

Segmentation with VLANs, hardened switch ports, and an access list that enforces a written policy. Lab 8 moves to Network Emulator and captures the traffic on a link, packet by packet.

- [All network labs](https://zeffiroh.github.io/ap-cybersecurity/labs.html)